<table>
    <tr>
        <td colspan=2><strong>International University of La Rioja (UNIR)</strong></td>
    </tr>
    <tr>
        <td colspan=2><strong>Master's Degree in Mathematical Engineering and Computing</strong></td>
    </tr>
    <tr>
        <td>Master's thesis submitted by:</td>
        <td><strong>Andrés Benavides, Diego Salamanca, Yeferson Salamanca.</strong></td>
    </tr>
    <tr>
        <td>Supervisor:</td>
        <td>PhD. Juan Francisco Martínez Cerda</td>
    </tr>
</table>


---


# 02 - Merge per-ticker CSV files into a single dataset

This notebook reads each `data/raw/<TICKER>.csv` produced by `01_download_data.ipynb`, keeps only the columns configured in `metadata/tickers.json`, outer-joins everything on the `Date` index, and then **cleans the result into a numeric, weekday-only dataset without missing values**, ready for preprocessing in `04_preprocess_data.ipynb`.

**Inputs**
- `data/raw/<TICKER>.csv` - one OHLCV CSV per ticker (sector + macro) written by notebook 01.
- `metadata/tickers.json` - sector / macro ticker lists and the columns to keep per group.

**Outputs** (in `data/processed/`)
- `merged_raw_data.csv` - the raw outer join of all tickers on `Date` (may contain `NaN`).
- `_merge_report.json` - a merge summary (shape, date range, non-null counts per ticker, anything skipped).
- `merged_processed_data.csv` - the **clean** file (numeric, weekdays only, holidays removed, missing values imputed). This is the file subsequent notebooks should consume.

**Columns kept per ticker (defined by `metadata/tickers.json`)**
- Sector tickers: `sectors_columns_to_keep` (currently `["Adj Close", "Volume"]`).
- Macro tickers: `macro_columns_to_keep` (currently `["Adj Close"]`).

**Workflow**
1. Imports and paths.
2. Load metadata and resolve the target sector / macro tickers.
3. Robust per-ticker loader (flattens the `yfinance` MultiIndex header and selects the configured columns).
4. Load all available tickers (sector tickers first, then macro tickers, so macro columns appear at the end).
5. Outer-join all tickers on the `Date` index (keep any date present in at least one ticker; missing values remain `NaN`).
6. Save `merged_raw_data.csv` and write `_merge_report.json`.
7. Date continuity check - coerce to numeric, remove weekends and weekday market holidays, and report unexpected missing values between consecutive trading days.
8. Detect and impute remaining missing values.
9. Save the clean numeric file as `merged_processed_data.csv`.

**Notes**
- Sector tickers are loaded first and macro tickers afterward, so macro columns are appended to the end of the merged CSV.
- `yfinance` saves single-ticker CSV files with a MultiIndex header (axis names `Price` / `Ticker`). This notebook detects that format and flattens it before selecting columns.
- Holiday rows are identified using **sector tickers only**: macro tickers (e.g. `^VIX` / `^TNX` / `DX-Y.NYB`) may trade on U.S. market holidays and would otherwise retain values in an otherwise empty row.

## 1. Imports and paths

In [1]:
import json
import logging
from pathlib import Path

import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("merge_data")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
METADATA_PATH = PROJECT_ROOT / "metadata" / "tickers.json"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MERGED_PATH = PROCESSED_DIR / "merged_raw_data.csv"
MERGE_REPORT_PATH = PROCESSED_DIR / "_merge_report.json"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT  : {PROJECT_ROOT}")
print(f"METADATA_PATH : {METADATA_PATH}")
print(f"RAW_DIR       : {RAW_DIR}")
print(f"MERGED_PATH   : {MERGED_PATH}")

PROJECT_ROOT  : /Applications/TFM_LSTM
METADATA_PATH : /Applications/TFM_LSTM/metadata/tickers.json
RAW_DIR       : /Applications/TFM_LSTM/data/raw
MERGED_PATH   : /Applications/TFM_LSTM/data/processed/merged_raw_data.csv


## 2. Load metadata and resolve target tickers

We extract four things from `tickers.json`:

1. `sectors_columns_to_keep` - the OHLCV columns to keep per sector ticker (default `["Adj Close", "Volume"]`).
2. `macro_columns_to_keep` - the columns to keep per macro ticker (default `["Adj Close"]`).
3. The complete flat list of sector tickers (across all sectors).
4. The list of macro tickers (under the top-level `macro` key).

Only tickers that actually have a CSV in `data/raw/` will be merged. Sector tickers are kept first and macro tickers afterward, so macro columns end up at the end of the merged file.

In [2]:
if not METADATA_PATH.exists():
    raise FileNotFoundError(
        f"Missing metadata file: {METADATA_PATH}. Run notebook 01 first."
    )

with METADATA_PATH.open("r", encoding="utf-8") as f:
    metadata = json.load(f)

sectors_columns_to_keep: list[str] = metadata.get("sectors_columns_to_keep", ["Adj Close", "Volume"])
macro_columns_to_keep: list[str] = metadata.get("macro_columns_to_keep", ["Adj Close"])

all_tickers: list[str] = [
    ticker
    for tickers in metadata["sectors"].values()
    for ticker in tickers.keys()
]
all_macro_tickers: list[str] = list(metadata.get("macro", {}).keys())

print(f"sectors_columns_to_keep : {sectors_columns_to_keep}")
print(f"macro_columns_to_keep   : {macro_columns_to_keep}")
print(f"Sector tickers in meta  : {len(all_tickers)}")
print(f"Sector tickers list     : {all_tickers}")
print(f"Macro tickers in meta   : {len(all_macro_tickers)}")
print(f"Macro tickers list      : {all_macro_tickers}")

sectors_columns_to_keep : ['Adj Close', 'Volume']
macro_columns_to_keep   : ['Adj Close']
Sector tickers in meta  : 25
Sector tickers list     : ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'LLY', 'UNH', 'JNJ', 'MRK', 'ISRG', 'DE', 'ADM', 'BG', 'MOS', 'CF', 'XOM', 'CVX', 'COP', 'EOG', 'SLB', 'JPM', 'BLK', 'GS', 'MS', 'AXP']
Macro tickers in meta   : 3
Macro tickers list      : ['^VIX', 'DX-Y.NYB', '^TNX']


## 3. Robust per-ticker loader

`load_ticker_csv` handles two on-disk formats:

- **yfinance MultiIndex CSV** (produced by `01_download_data.ipynb`). The header has three rows:
    1. `Price,Adj Close,Close,High,Low,Open,Volume`
    2. `Ticker,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL`
    3. `Date,,,,,,`

    We read the first two rows as the column header, skip the third (the row containing the `Date` axis name), and keep only level 0 of the MultiIndex (the actual column names such as `Adj Close`, `Volume`).
- **Flat CSV** (`Date,Open,High,Low,Close,Adj Close,Volume`) - one header row, with `Date` as the index.

After loading, we keep only the configured columns and rename them as `<TICKER>_<COLUMN>` so they remain unique after merging.

In [3]:
def _is_multiindex_csv(path: Path) -> bool:
    """Detect yfinance's MultiIndex CSV format by looking at the first line."""
    with path.open("r", encoding="utf-8") as f:
        first = f.readline().strip()
    return first.startswith("Price,") or first.startswith("\"Price\",")


def load_ticker_csv(path: Path, ticker: str, columns: list[str]) -> pd.DataFrame:
    """Load a single ticker CSV and return a DataFrame with the configured columns.

    The returned DataFrame has a DatetimeIndex named ``Date`` and columns renamed
    to ``<TICKER>_<COLUMN>`` so they stay unique after merging across tickers.
    """
    if _is_multiindex_csv(path):
        df = pd.read_csv(
            path,
            header=[0, 1],
            skiprows=[2],
            index_col=0,
            parse_dates=True,
        )
        df.columns = df.columns.get_level_values(0)
    else:
        df = pd.read_csv(path, index_col=0, parse_dates=True)

    df.index.name = "Date"
    df = df.sort_index()

    missing = [c for c in columns if c not in df.columns]
    if missing:
        logger.warning(
            "[%s] missing columns %s; available: %s",
            ticker, missing, list(df.columns),
        )
    keep = [c for c in columns if c in df.columns]
    df = df[keep].copy()

    df.columns = [f"{ticker}_{c}" for c in df.columns]
    return df

## 4. Load all available tickers

We iterate over the tickers in the metadata - sector tickers first (using `sectors_columns_to_keep`), then macro tickers (using `macro_columns_to_keep`) - and only attempt to load those that actually have a CSV in `data/raw/`. Anything missing is logged and skipped.

Because the DataFrames are stored in a dictionary that preserves insertion order, loading sector tickers before macro tickers ensures macro columns end up at the end of the merged file.

In [4]:
frames: dict[str, pd.DataFrame] = {}
missing_csvs: list[str] = []
load_errors: list[dict] = []

def _load_group(tickers: list[str], columns: list[str], group_label: str) -> None:
    """Load every ticker in ``tickers`` into ``frames`` using ``columns``."""
    for ticker in tickers:
        csv_path = RAW_DIR / f"{ticker}.csv"
        if not csv_path.exists():
            logger.warning("[%s/%s] no CSV at %s - skipping", group_label, ticker, csv_path)
            missing_csvs.append(ticker)
            continue
        try:
            frames[ticker] = load_ticker_csv(csv_path, ticker, columns)
            logger.info(
                "[%s/%s] loaded %d rows, columns=%s",
                group_label, ticker, len(frames[ticker]), list(frames[ticker].columns),
            )
        except Exception as exc:
            logger.error("[%s/%s] failed to load: %s", group_label, ticker, exc)
            load_errors.append({"ticker": ticker, "group": group_label, "error": str(exc)})

_load_group(all_tickers, sectors_columns_to_keep, "sector")
_load_group(all_macro_tickers, macro_columns_to_keep, "macro")

loaded_sector = [t for t in all_tickers if t in frames]
loaded_macro = [t for t in all_macro_tickers if t in frames]

print(f"\nLoaded sectors : {len(loaded_sector)} / {len(all_tickers)}")
print(f"Loaded macros  : {len(loaded_macro)} / {len(all_macro_tickers)}")
print(f"Missing        : {missing_csvs}")
print(f"Errors         : {load_errors}")

08:37:06 | INFO | [sector/AAPL] loaded 3018 rows, columns=['AAPL_Adj Close', 'AAPL_Volume']


08:37:06 | INFO | [sector/MSFT] loaded 3018 rows, columns=['MSFT_Adj Close', 'MSFT_Volume']
08:37:06 | INFO | [sector/NVDA] loaded 3018 rows, columns=['NVDA_Adj Close', 'NVDA_Volume']
08:37:06 | INFO | [sector/GOOGL] loaded 3018 rows, columns=['GOOGL_Adj Close', 'GOOGL_Volume']
08:37:06 | INFO | [sector/AMZN] loaded 3018 rows, columns=['AMZN_Adj Close', 'AMZN_Volume']
08:37:06 | INFO | [sector/LLY] loaded 3018 rows, columns=['LLY_Adj Close', 'LLY_Volume']
08:37:06 | INFO | [sector/UNH] loaded 3018 rows, columns=['UNH_Adj Close', 'UNH_Volume']
08:37:06 | INFO | [sector/JNJ] loaded 3018 rows, columns=['JNJ_Adj Close', 'JNJ_Volume']
08:37:06 | INFO | [sector/MRK] loaded 3018 rows, columns=['MRK_Adj Close', 'MRK_Volume']
08:37:06 | INFO | [sector/ISRG] loaded 3018 rows, columns=['ISRG_Adj Close', 'ISRG_Volume']
08:37:06 | INFO | [sector/DE] loaded 3018 rows, columns=['DE_Adj Close', 'DE_Volume']
08:37:06 | INFO | [sector/ADM] loaded 3018 rows, columns=['ADM_Adj Close', 'ADM_Volume']
08:37:


Loaded sectors : 25 / 25
Loaded macros  : 3 / 3
Missing        : []
Errors         : []


## 5. Outer-join all tickers on the `Date` index

We use `pd.concat(..., axis=1, join="outer")`, which is equivalent to a chained outer join on the index. Any date present in at least one ticker is retained; missing values remain `NaN` (empty cells in the resulting CSV).

In [5]:
if not frames:
    raise RuntimeError(
        "No ticker CSVs were loaded; run 01_download_data.ipynb first."
    )

merged = pd.concat(frames.values(), axis=1, join="outer", sort=False).sort_index()
merged.index.name = "Date"

print(f"Merged shape    : {merged.shape}")
print(f"Date range      : {merged.index.min()} -> {merged.index.max()}")
print(f"# columns       : {len(merged.columns)}")
print("First columns   :", list(merged.columns[:6]))
merged.head()

Merged shape    : (3023, 53)
Date range      : 2014-05-29 00:00:00 -> 2026-05-29 00:00:00
# columns       : 53
First columns   : ['AAPL_Adj Close', 'AAPL_Volume', 'MSFT_Adj Close', 'MSFT_Volume', 'NVDA_Adj Close', 'NVDA_Volume']


,AAPL_Adj Close,AAPL_Volume,MSFT_Adj Close,MSFT_Volume,NVDA_Adj Close,NVDA_Volume,GOOGL_Adj Close,GOOGL_Volume,AMZN_Adj Close,AMZN_Volume,...,BLK_Volume,GS_Adj Close,GS_Volume,MS_Adj Close,MS_Volume,AXP_Adj Close,AXP_Volume,^VIX_Adj Close,DX-Y.NYB_Adj Close,^TNX_Adj Close
Date,,,,,,,,,,,,,,,,,,,,,
2014-05-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.57,NaN,2.447
2014-05-30,19.825077,564020800.0,34.310696,34567600.0,0.451953,239128000.0,28.348204,37434000.0,15.6275,85204000.0,...,365100.0,127.069298,2206300.0,22.510998,7634400.0,77.304619,3594800.0,11.40,80.370003,2.457
2014-06-02,19.688843,369350800.0,34.184971,18504300.0,0.450525,181500000.0,27.985695,33210000.0,15.4420,44068000.0,...,404200.0,127.244278,2162500.0,22.540169,7509400.0,77.634140,2832100.0,11.58,80.639999,2.534
2014-06-03,19.967274,292709200.0,33.765945,18068900.0,0.448622,243212000.0,27.498230,40626000.0,15.3595,47584000.0,...,719300.0,128.651566,2408600.0,22.503696,10079500.0,77.498924,2589100.0,11.87,80.550003,2.593
2014-06-04,20.195278,335482000.0,33.791069,23209000.0,0.449098,161236000.0,27.461033,34582000.0,15.3390,42142000.0,...,558200.0,129.120773,2098400.0,22.759003,8074700.0,77.566490,2637000.0,12.08,80.669998,2.606


## 6. Save the merged file and write a merge report

The merged DataFrame is written to `data/processed/merged_raw_data.csv` with `Date` as the first column. A small JSON report is also saved alongside it, summarizing the merge (shape, date range, non-null counts per ticker, anything skipped).

In [6]:
merged.to_csv(MERGED_PATH, index=True)

nan_per_column = merged.isna().sum()
non_null_per_column = merged.notna().sum()

total_expected = len(all_tickers) + len(all_macro_tickers)

report = {
    "sectors_columns_to_keep": sectors_columns_to_keep,
    "macro_columns_to_keep": macro_columns_to_keep,
    "sector_tickers_in_metadata": all_tickers,
    "macro_tickers_in_metadata": all_macro_tickers,
    "sector_tickers_merged": loaded_sector,
    "macro_tickers_merged": loaded_macro,
    "missing_csvs": missing_csvs,
    "load_errors": load_errors,
    "rows": int(merged.shape[0]),
    "columns": int(merged.shape[1]),
    "date_min": str(merged.index.min()),
    "date_max": str(merged.index.max()),
    "non_null_per_column": {c: int(v) for c, v in non_null_per_column.items()},
    "nan_per_column": {c: int(v) for c, v in nan_per_column.items()},
    "output_path": str(MERGED_PATH),
}
MERGE_REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")

print("=" * 60)
print("MERGE REPORT")
print("=" * 60)
print(f"Sectors merged   : {len(loaded_sector)} / {len(all_tickers)}")
print(f"Macros merged    : {len(loaded_macro)} / {len(all_macro_tickers)}")
print(f"Total merged     : {len(frames)} / {total_expected}")
print(f"Missing CSVs     : {missing_csvs}")
print(f"Load errors      : {load_errors}")
print(f"Rows x columns   : {merged.shape[0]} x {merged.shape[1]}")
print(f"Date range       : {merged.index.min()} -> {merged.index.max()}")
print(f"Last columns     : {list(merged.columns[-len(loaded_macro) * len(macro_columns_to_keep):]) if loaded_macro else []}")
print("-" * 60)
print(f"Saved merged CSV : {MERGED_PATH}")
print(f"Saved report     : {MERGE_REPORT_PATH}")

MERGE REPORT
Sectors merged   : 25 / 25
Macros merged    : 3 / 3
Total merged     : 28 / 28
Missing CSVs     : []
Load errors      : []
Rows x columns   : 3023 x 53
Date range       : 2014-05-29 00:00:00 -> 2026-05-29 00:00:00
Last columns     : ['^VIX_Adj Close', 'DX-Y.NYB_Adj Close', '^TNX_Adj Close']
------------------------------------------------------------
Saved merged CSV : /Applications/TFM_LSTM/data/processed/merged_raw_data.csv
Saved report     : /Applications/TFM_LSTM/data/processed/_merge_report.json


## 7. Check date continuity

The merged file contains financial market data, so valid observations should only fall on weekdays (Monday through Friday). In this section:

1. We coerce all values to numeric (`pd.to_numeric(errors="coerce")`) so the output file contains only numbers (anything that cannot be parsed becomes `NaN` and is handled in section 8).
2. We remove any row whose index falls on Saturday or Sunday.
3. We remove **weekday holidays**. On a U.S. market holiday, exchanges are closed, so sector tickers have no observation. A row can still slip in when a macro ticker (e.g. `^TNX`, `^VIX`, `DX-Y.NYB`) traded that day. We therefore remove any weekday row where **most sector tickers are empty/null**, evaluating *only* sector tickers and ignoring macro tickers. A sector ticker counts as missing on a date when **all** its columns are `NaN`.
4. We inspect the spacing between consecutive weekday rows. Within Monday-Friday, the expected interval is **1 business day** (with the usual Fri -> Mon jump of 3 calendar days). Anything larger is reported because it usually indicates a market holiday or a download gap.

In [7]:
PROCESSED_PATH = PROCESSED_DIR / "merged_processed_data.csv"

clean = merged.apply(pd.to_numeric, errors="coerce").copy()
clean.index = pd.to_datetime(clean.index)
clean = clean.sort_index()

weekday_codes = clean.index.weekday
weekend_mask = weekday_codes >= 5
n_weekend = int(weekend_mask.sum())

print("=" * 60)
print("WEEKDAY FILTER")
print("=" * 60)
print(f"Rows before filtering : {len(clean)}")
if n_weekend:
    weekend_dates = clean.index[weekend_mask]
    print(f"Weekend rows found    : {n_weekend} (dropping Saturday/Sunday)")
    preview = ", ".join(d.strftime("%Y-%m-%d (%a)") for d in weekend_dates[:5])
    if n_weekend > 5:
        preview += f", ... (+{n_weekend - 5} more)"
    print(f"Examples              : {preview}")
else:
    print("Weekend rows found    : 0")

clean = clean[~weekend_mask]
print(f"Rows after filtering  : {len(clean)}")

print()
print("=" * 60)
print("HOLIDAY FILTER (drop weekdays where most sector tickers are empty)")
print("=" * 60)

# Build the per-ticker null mask using ONLY sector tickers (macro tickers
# such as ^VIX / ^TNX / DX-Y.NYB are ignored: they can trade on US market
# holidays and would otherwise keep an otherwise-empty row alive).
sector_null_by_ticker = pd.DataFrame(
    {
        ticker: clean[list(frames[ticker].columns)].isna().all(axis=1)
        for ticker in loaded_sector
    },
    index=clean.index,
)

n_sector = len(loaded_sector)
null_sector_count = sector_null_by_ticker.sum(axis=1)
# "most" -> strictly more than half of the sector tickers are empty/null.
holiday_mask = null_sector_count > (n_sector / 2)
n_holiday = int(holiday_mask.sum())

print(f"Sector tickers considered : {n_sector} (macro tickers ignored)")
print(f"Rows before filtering     : {len(clean)}")
if n_holiday:
    holiday_dates = clean.index[holiday_mask]
    print(f"Holiday rows found        : {n_holiday} (most sector tickers empty/null)")
    preview = ", ".join(
        f"{d.strftime('%Y-%m-%d (%a)')} [{int(null_sector_count[d])}/{n_sector} empty]"
        for d in holiday_dates[:5]
    )
    if n_holiday > 5:
        preview += f", ... (+{n_holiday - 5} more)"
    print(f"Examples                  : {preview}")
else:
    print("Holiday rows found        : 0")

clean = clean[~holiday_mask]
print(f"Rows after filtering      : {len(clean)}")

print()
print("-" * 60)
print("WEEKDAY GAP CHECK (expected 1 business day between rows)")
print("-" * 60)

prev_idx = clean.index[:-1]
curr_idx = clean.index[1:]
actual_days = (curr_idx - prev_idx).days
expected_days = [3 if wd == 4 else 1 for wd in prev_idx.weekday]

gaps = [
    (prev_idx[i], curr_idx[i], int(actual_days[i]), int(expected_days[i]))
    for i in range(len(curr_idx))
    if int(actual_days[i]) > int(expected_days[i])
]

if gaps:
    print(f"Found {len(gaps)} weekday gaps with interval greater than expected:")
    for prev, curr, actual, expected in gaps[:20]:
        print(
            f"  {prev.strftime('%Y-%m-%d (%a)')} -> {curr.strftime('%Y-%m-%d (%a)')}"
            f" : {actual} calendar days (expected {expected})"
        )
    if len(gaps) > 20:
        print(f"  ... and {len(gaps) - 20} more.")
    print("Note: these are usually market holidays (e.g. Christmas, July 4th).")
else:
    print("All consecutive weekday rows are 1 business day apart - no gaps.")

WEEKDAY FILTER
Rows before filtering : 3023
Weekend rows found    : 0
Rows after filtering  : 3023

HOLIDAY FILTER (drop weekdays where most sector tickers are empty)
Sector tickers considered : 25 (macro tickers ignored)
Rows before filtering     : 3023
Holiday rows found        : 5 (most sector tickers empty/null)
Examples                  : 2014-05-29 (Thu) [25/25 empty], 2016-01-18 (Mon) [25/25 empty], 2025-01-09 (Thu) [25/25 empty], 2025-05-26 (Mon) [25/25 empty], 2026-05-25 (Mon) [25/25 empty]
Rows after filtering      : 3018

------------------------------------------------------------
WEEKDAY GAP CHECK (expected 1 business day between rows)
------------------------------------------------------------
Found 113 weekday gaps with interval greater than expected:
  2014-07-03 (Thu) -> 2014-07-07 (Mon) : 4 calendar days (expected 1)
  2014-08-29 (Fri) -> 2014-09-02 (Tue) : 4 calendar days (expected 3)
  2014-11-26 (Wed) -> 2014-11-28 (Fri) : 2 calendar days (expected 1)
  2014-12-24

## 8. Detect and impute missing values

The remaining columns contain `Adj Close` prices and `Volume` figures. We:

1. Count missing/empty values per column and report nonzero counts.
2. Fill missing values with the **average of neighboring days** using `DataFrame.interpolate(method="linear")`. For an isolated `NaN`, linear interpolation between the preceding and following observations is exactly the mean of those two values, matching the "average of surrounding days" rule.
3. For any `NaN` remaining at the start or end of a series (where interpolation has no neighbor on one side), fall back to the nearest available value (`limit_direction="both"`).

In [8]:
print("=" * 60)
print("MISSING VALUE CHECK")
print("=" * 60)

missing_before = clean.isna().sum()
total_missing_before = int(missing_before.sum())
cols_with_missing = missing_before[missing_before > 0].sort_values(ascending=False)

print(f"Total missing cells before imputation : {total_missing_before}")
if total_missing_before:
    print(f"Columns with missing values ({len(cols_with_missing)}):")
    for col, n in cols_with_missing.items():
        print(f"  {col:<25s} : {int(n)} NaN")
else:
    print("No missing values found in Adj Close / Volume columns.")

clean = clean.interpolate(method="linear", limit_direction="both")

missing_after = clean.isna().sum()
total_missing_after = int(missing_after.sum())

print()
print(f"Total missing cells after imputation  : {total_missing_after}")
if total_missing_after:
    print("Remaining NaN per column (could not be interpolated):")
    for col, n in missing_after[missing_after > 0].items():
        print(f"  {col:<25s} : {int(n)} NaN")
else:
    print("All gaps were filled using the average of the surrounding days.")

MISSING VALUE CHECK
Total missing cells before imputation : 3
Columns with missing values (2):
  DX-Y.NYB_Adj Close        : 2 NaN
  ^TNX_Adj Close            : 1 NaN

Total missing cells after imputation  : 0
All gaps were filled using the average of the surrounding days.


## 9. Save the clean numeric merged file

The fully numeric, weekday-only, imputed DataFrame is saved in `data/processed/merged_processed_data.csv`. This is the file subsequent notebooks should consume.

In [9]:
clean.to_csv(PROCESSED_PATH, index=True)

non_numeric_cols = [c for c in clean.columns if not pd.api.types.is_numeric_dtype(clean[c])]

print("=" * 60)
print("PROCESSED FILE SAVED")
print("=" * 60)
print(f"Path             : {PROCESSED_PATH}")
print(f"Shape            : {clean.shape[0]} rows x {clean.shape[1]} columns")
print(f"Date range       : {clean.index.min().date()} -> {clean.index.max().date()}")
print(f"All numeric?     : {len(non_numeric_cols) == 0} (non-numeric columns: {non_numeric_cols})")
print(f"Any NaN left?    : {bool(clean.isna().any().any())}")
clean.head()

PROCESSED FILE SAVED
Path             : /Applications/TFM_LSTM/data/processed/merged_processed_data.csv
Shape            : 3018 rows x 53 columns
Date range       : 2014-05-30 -> 2026-05-29
All numeric?     : True (non-numeric columns: [])
Any NaN left?    : False


,AAPL_Adj Close,AAPL_Volume,MSFT_Adj Close,MSFT_Volume,NVDA_Adj Close,NVDA_Volume,GOOGL_Adj Close,GOOGL_Volume,AMZN_Adj Close,AMZN_Volume,...,BLK_Volume,GS_Adj Close,GS_Volume,MS_Adj Close,MS_Volume,AXP_Adj Close,AXP_Volume,^VIX_Adj Close,DX-Y.NYB_Adj Close,^TNX_Adj Close
Date,,,,,,,,,,,,,,,,,,,,,
2014-05-30,19.825077,564020800.0,34.310696,34567600.0,0.451953,239128000.0,28.348204,37434000.0,15.627500,85204000.0,...,365100.0,127.069298,2206300.0,22.510998,7634400.0,77.304619,3594800.0,11.40,80.370003,2.457
2014-06-02,19.688843,369350800.0,34.184971,18504300.0,0.450525,181500000.0,27.985695,33210000.0,15.442000,44068000.0,...,404200.0,127.244278,2162500.0,22.540169,7509400.0,77.634140,2832100.0,11.58,80.639999,2.534
2014-06-03,19.967274,292709200.0,33.765945,18068900.0,0.448622,243212000.0,27.498230,40626000.0,15.359500,47584000.0,...,719300.0,128.651566,2408600.0,22.503696,10079500.0,77.498924,2589100.0,11.87,80.550003,2.593
2014-06-04,20.195278,335482000.0,33.791069,23209000.0,0.449098,161236000.0,27.461033,34582000.0,15.339000,42142000.0,...,558200.0,129.120773,2098400.0,22.759003,8074700.0,77.566490,2637000.0,12.08,80.669998,2.606
2014-06-05,20.274511,303805600.0,34.536945,31865200.0,0.451001,188696000.0,28.014956,35868000.0,16.178499,155934000.0,...,566400.0,129.271790,1859600.0,23.072668,8769400.0,78.402916,3219200.0,11.68,80.370003,2.584
